# Text & NLP with Skyulf — Hands-on with Real Data

This notebook teaches the **text / NLP nodes** in `skyulf-core` from scratch, using a **real scikit-learn dataset** (the *20 Newsgroups* corpus). No prior NLP knowledge is required.

## The one idea you must understand

Machine-learning models only understand **numbers**, not words. So every text pipeline has the same shape:

```
raw text  ->  [ turn text into numbers ]  ->  a model
```

The "turn text into numbers" step is called **vectorization**. Skyulf gives you several vectorizers, each a `Calculator` (learns from the training data) + `Applier` (transforms any data the same way). This notebook walks through each one, explains *when to use it*, and then plugs it into a classifier.

## What you'll do

| # | Section | Node | What it does |
|---|---------|------|--------------|
| 1 | Load data | — | Real newsgroup posts from scikit-learn |
| 2 | Count Vectorizer | `count_vectorizer` | Bag-of-words: how many times each word appears |
| 3 | TF-IDF Vectorizer | `tfidf_vectorizer` | Like counts, but down-weights common words |
| 4 | Hashing Vectorizer | `hashing_vectorizer` | Same idea, no vocabulary stored (memory-cheap) |
| 5 | Tokenizer | `tokenizer` | Clean/split text *before* vectorizing (chaining) |
| 6 | Model combinations | Naive Bayes / Logistic Regression | Put it all together and measure accuracy |
| 7 | Sentence Embedder | `sentence_embedder` | Modern dense embeddings (optional dependency) |

> **Every node follows the same two-step pattern:** `artifact = Calculator().fit(df, config)` then `out = Applier().apply(df, artifact)`. Once you learn one, you know them all.

## 1. Load real text data

We use **20 Newsgroups** — ~18k forum posts labelled by topic. We pick 4 topics so it's a quick, clear classification problem. scikit-learn even ships a ready-made train/test split, so we don't have to split it ourselves.

We strip headers/footers/signatures (`remove=...`) so the model has to learn from the *actual words*, not metadata leaks — this is the honest way to benchmark text models.

> The first run downloads ~14 MB and caches it locally. If you're offline, the cell falls back to a tiny built-in sample so the rest of the notebook still runs (accuracies will just be toy numbers).

In [1]:
import warnings

warnings.filterwarnings("ignore")

import pandas as pd
from sklearn.metrics import accuracy_score

CATEGORIES = ["rec.sport.baseball", "sci.med", "comp.graphics", "talk.politics.guns"]
USING_REAL_DATA = True

try:
    from sklearn.datasets import fetch_20newsgroups

    def _load(subset):
        bunch = fetch_20newsgroups(
            subset=subset,
            categories=CATEGORIES,
            remove=("headers", "footers", "quotes"),
            shuffle=True,
            random_state=42,
        )
        return pd.DataFrame({"text": bunch.data, "label": bunch.target}), bunch.target_names

    train_df, target_names = _load("train")
    test_df, _ = _load("test")
    # Drop empty posts left behind after stripping quotes/headers.
    train_df = train_df[train_df["text"].str.strip().str.len() > 0].reset_index(drop=True)
    test_df = test_df[test_df["text"].str.strip().str.len() > 0].reset_index(drop=True)
except Exception as exc:  # offline / download blocked
    USING_REAL_DATA = False
    print("Could not download 20 Newsgroups, using a tiny offline sample:", exc)
    target_names = ["sport", "medicine"]
    train_df = pd.DataFrame(
        {
            "text": [
                "the team won the game last night great pitching",
                "baseball season starts the home run was amazing",
                "the doctor prescribed medicine for the infection",
                "patient recovery after surgery and treatment",
                "the coach and players celebrated the championship",
                "clinical trial showed the drug reduced symptoms",
            ],
            "label": [0, 0, 1, 1, 0, 1],
        }
    )
    test_df = pd.DataFrame(
        {
            "text": [
                "the pitcher threw a no hitter in the game",
                "the hospital treated the patient with new medicine",
            ],
            "label": [0, 1],
        }
    )

print(f"Train rows: {len(train_df)}   Test rows: {len(test_df)}")
print(f"Classes: {list(enumerate(target_names))}")
train_df.head(3)

Train rows: 2256   Test rows: 1503
Classes: [(0, 'comp.graphics'), (1, 'rec.sport.baseball'), (2, 'sci.med'), (3, 'talk.politics.guns')]


,text,label
0,I am 35 and am recovering from a case of Chick...,2
1,I'm looking for software (hopefully free and r...,1
2,"\nI'm still catching up from Spring Break, but...",1


## 2. Count Vectorizer — "bag of words"

**The simplest vectorizer.** It builds a vocabulary of every word it sees in training, then represents each document as *how many times each word appears*. Word order is ignored — hence "bag" of words.

```
"the cat sat"  ->  the:1  cat:1  sat:1   (one column per vocabulary word)
```

**Config knobs that matter:**
- `max_features` — keep only the *N* most frequent words (controls width / memory).
- `min_df` — ignore words that appear in fewer than this many documents (drops rare noise/typos).
- `ngram_range` — `[1, 1]` = single words; `[1, 2]` = also count 2-word phrases ("home run").

**Use it when:** you want a simple, interpretable baseline and you can read the vocabulary.

Below we cap to 30 features just so the vocabulary is small enough to print.

In [2]:
from skyulf.preprocessing.vectorization import (
    CountVectorizerApplier,
    CountVectorizerCalculator,
)

count_cfg = {
    "columns": ["text"],
    "max_features": 30,  # tiny vocab so we can print it
    "min_df": 5,  # word must appear in >= 5 posts
    "ngram_range": [1, 1],
}

# Step 1 - LEARN the vocabulary from the training set
cv_art = CountVectorizerCalculator().fit(train_df, count_cfg)

print("Learned vocabulary (column index -> word):")
for word, idx in sorted(cv_art["vocabulary"].items(), key=lambda kv: kv[1]):
    print(f"  [{idx:>2}] {word}")

# Step 2 - APPLY: turn text into count columns
train_counts = CountVectorizerApplier().apply(train_df, cv_art)
print("\nOutput shape (rows x columns):", train_counts.shape)
train_counts[cv_art["output_columns"][:8]].head(3)

Learned vocabulary (column index -> word):
  [ 0] an
  [ 1] and
  [ 2] are
  [ 3] as
  [ 4] at
  [ 5] be
  [ 6] but
  [ 7] by
  [ 8] can
  [ 9] for
  [10] from
  [11] have
  [12] he
  [13] if
  [14] in
  [15] is
  [16] it
  [17] not
  [18] of
  [19] on
  [20] or
  [21] that
  [22] the
  [23] they
  [24] this
  [25] to
  [26] was
  [27] with
  [28] would
  [29] you

Output shape (rows x columns): (2256, 32)


,text__count__an,text__count__and,text__count__are,text__count__as,text__count__at,text__count__be,text__count__but,text__count__by
0,0,2,1,0,1,0,0,0
1,0,1,0,0,0,0,0,1
2,1,14,5,1,3,2,4,0


## 3. TF-IDF Vectorizer — counts, but smarter

Plain counts have a problem: very common words ("the", "is", "and") get big counts but carry almost no meaning. **TF-IDF** fixes this by multiplying two things:

- **TF** (term frequency) — how often the word appears in *this* document.
- **IDF** (inverse document frequency) — how *rare* the word is across *all* documents.

The result: words that are frequent **here** but rare **overall** (the distinctive ones) get the highest weight. This is the **single most common starting point** for text classification.

**Extra knobs:**
- `ngram_range: [1, 2]` — include 2-word phrases, which often help.
- `sublinear_tf: true` — dampen runaway counts with `1 + log(tf)` (usually a small win).

Here we keep up to 2000 features and unigrams+bigrams — a solid, realistic config.

In [3]:
from skyulf.preprocessing.vectorization import (
    TfidfVectorizerApplier,
    TfidfVectorizerCalculator,
)

tfidf_cfg = {
    "columns": ["text"],
    "max_features": 2000,
    "min_df": 2,
    "ngram_range": [1, 2],  # unigrams + bigrams
    "sublinear_tf": True,
}

tf_art = TfidfVectorizerCalculator().fit(train_df, tfidf_cfg)
tf_app = TfidfVectorizerApplier()

# Fit on train, then apply the SAME learned weights to both train and test.
train_tfidf = tf_app.apply(train_df, tf_art)
test_tfidf = tf_app.apply(test_df, tf_art)
tf_cols = tf_art["output_columns"]

print("TF-IDF features:", len(tf_cols))
print("A few feature columns:", tf_cols[:6])
# Show the highest-weighted words in the first post
row0 = train_tfidf.iloc[0]
top = row0[tf_cols].astype(float).sort_values(ascending=False).head(8)
print("\nMost important words in post #0:")
print(top.to_string())

TF-IDF features: 2000
A few feature columns: ['text__tfidf__00', 'text__tfidf__000', 'text__tfidf__01', 'text__tfidf__02', 'text__tfidf__03', 'text__tfidf__04']

Most important words in post #0:
text__tfidf__am           0.239910
text__tfidf__over         0.203720
text__tfidf__year old     0.194868
text__tfidf__speed        0.189485
text__tfidf__there any    0.189485
text__tfidf__longer       0.187534
text__tfidf__my           0.187094
text__tfidf__when they    0.184786


## 4. Hashing Vectorizer — no vocabulary needed

Both vectorizers above must **store a vocabulary** (every word -> column). For huge or streaming datasets that table can get big. The **hashing trick** skips it: it runs each word through a hash function to decide its column directly. Nothing is stored.

**Trade-offs:**
- Constant, tiny memory; great for very large or online data.
- Stateless — `fit` learns nothing, so train/serve can't drift.
- Not interpretable (you can't map a column back to a word).
- Rare hash *collisions* (two words share a column).

**Config:** `n_features` = number of hash buckets (columns). Bigger = fewer collisions. Powers of two (1024, 2048, ...) are conventional.

**Use it when:** the dataset is too large to hold a vocabulary, or you need a fixed-width, stateless transform.

In [4]:
from skyulf.preprocessing.vectorization import (
    HashingVectorizerApplier,
    HashingVectorizerCalculator,
)

hash_cfg = {"columns": ["text"], "n_features": 1024, "norm": "l2"}

h_art = HashingVectorizerCalculator().fit(train_df, hash_cfg)
train_hash = HashingVectorizerApplier().apply(train_df, h_art)

print("Hashing output shape:", train_hash.shape)
print("Vocabulary stored?", "vocabulary" in h_art, "(hashing keeps none)")
print("Always exactly n_features feature columns:", len(h_art["output_columns"]))

Hashing output shape: (2256, 1026)
Vocabulary stored? False (hashing keeps none)
Always exactly n_features feature columns: 1024


## 5. Tokenizer — clean the text *before* vectorizing

Sometimes you want to pre-process text **once** and reuse the result. The **Tokenizer** splits text into tokens and can drop noise — for example removing English **stop words** ("the", "and", "is"...). It's *stateless* and outputs a cleaned, space-joined string column named `<col>__tokens` (plus an optional `<col>__token_count`).

This demonstrates a key Skyulf pattern: **chaining nodes**. The output of one node becomes the input column of the next:

```
text  ->  Tokenizer (drop stop words)  ->  text__tokens  ->  TF-IDF  ->  model
```

**Use it when:** you want explicit, inspectable cleaning, or you want to share one cleaned column across several downstream vectorizers.

In [5]:
from skyulf.preprocessing.vectorization import (
    TokenizerApplier,
    TokenizerCalculator,
)

tok_cfg = {
    "columns": ["text"],
    "analyzer": "word",
    "stop_words": "english",  # drop common filler words
    "add_token_count": True,
}

tok_art = TokenizerCalculator().fit(train_df, tok_cfg)
tok_app = TokenizerApplier()
train_tok = tok_app.apply(train_df, tok_art)
test_tok = tok_app.apply(test_df, tok_art)

print("New columns added:", tok_art["output_columns"])
print("\nCleaned tokens (stop words removed):")
for i in range(min(2, len(train_tok))):
    txt = str(train_tok["text__tokens"].iloc[i])[:90]
    n = train_tok["text__token_count"].iloc[i]
    print(f"  [{n:>3} tokens] {txt}...")

New columns added: ['text__tokens', 'text__token_count']

Cleaned tokens (stop words removed):
  [ 31 tokens] 35 recovering case chicken pox contracted year old daughter quite little puppies bod point...
  [ 19 tokens] looking software hopefully free runs unix box track statistics company softball team batti...


In [6]:
# Chain it: vectorize the CLEANED tokens instead of the raw text.
chain_cfg = {
    "columns": ["text__tokens"],  # <- the Tokenizer's output column
    "max_features": 2000,
    "min_df": 2,
    "ngram_range": [1, 2],
    "sublinear_tf": True,
}
chain_art = TfidfVectorizerCalculator().fit(train_tok, chain_cfg)
chain_app = TfidfVectorizerApplier()
train_chain = chain_app.apply(train_tok, chain_art)
test_chain = chain_app.apply(test_tok, chain_art)
chain_cols = chain_art["output_columns"]
print("Tokenizer -> TF-IDF produced", len(chain_cols), "features from pre-cleaned text.")

Tokenizer -> TF-IDF produced 2000 features from pre-cleaned text.


## 6. Model combinations — putting it together

Now we plug each vectorized representation into a classifier and **measure test accuracy**. This is where you see *which combination wins* on real data.

### Can I use *any* model, or only specific ones?

**Once text is vectorized it is just numeric columns** — so you can feed it to **any classifier or regressor in Skyulf** exactly like ordinary tabular data: `logistic_regression`, `random_forest_classifier`, `svc`, `gradient_boosting_classifier`, `hist_gradient_boosting_classifier`, `extra_trees_classifier`, `k_neighbors_classifier`, `decision_tree_classifier`, `adaboost_classifier`, `xgboost_classifier`, `lgbm_classifier`, the Naive Bayes family, and the ensemble nodes (`voting_classifier`, `stacking_classifier`). For regression targets the matching `*_regressor` nodes work too.

There is only **one real constraint**, and it comes from the *model*, not from Skyulf:

| Features | Works with | Avoid | Why |
|----------|-----------|-------|-----|
| Counts / TF-IDF (non-negative) | **Multinomial NB**, Logistic Regression, Linear SVC, trees, boosting | — | NB is the fast, strong text baseline; linear models love high-dimensional sparse text |
| Binary word presence | **Bernoulli NB**, anything above | — | Bernoulli models "word present / absent" |
| Dense embeddings (can be negative) | Logistic Regression, SVC, Random Forest, boosting | **Multinomial / Bernoulli NB** | Naive Bayes requires **non-negative** inputs; embeddings have negatives |

> **Rule of thumb:** start with **TF-IDF + Multinomial Naive Bayes** (cheap and surprisingly strong), then try **Logistic Regression** for a robust linear baseline, and reach for tree ensembles / boosting when you have lots of data and want the extra accuracy. The *only* pairing to avoid is Naive Bayes on embeddings.

The helper below trains a model and reports accuracy so we can compare combos fairly.

In [7]:
from skyulf.modeling.classification import (
    LogisticRegressionApplier,
    LogisticRegressionCalculator,
)
from skyulf.modeling.naive_bayes import (
    BernoulliNBApplier,
    BernoulliNBCalculator,
    MultinomialNBApplier,
    MultinomialNBCalculator,
)

results = {}


def evaluate(name, calc, applier, train_X, test_X, cfg=None):
    """Fit a classifier on train_X (labels from train_df) and score on test_X."""
    model_art = calc.fit(train_X, train_df["label"], config=cfg or {})
    preds = applier.predict(test_X, model_art)
    acc = accuracy_score(test_df["label"], preds)
    results[name] = acc
    print(f"{name:<40} accuracy = {acc:.3f}")
    return acc


# Classic strong baseline
evaluate(
    "TF-IDF + MultinomialNB",
    MultinomialNBCalculator(),
    MultinomialNBApplier(),
    train_tfidf[tf_cols],
    test_tfidf[tf_cols],
    {"alpha": 0.1},
)

# Same features, different model
evaluate(
    "TF-IDF + LogisticRegression",
    LogisticRegressionCalculator(),
    LogisticRegressionApplier(),
    train_tfidf[tf_cols],
    test_tfidf[tf_cols],
    {"max_iter": 1000},
)

# Mixed pipeline: cleaned tokens -> TF-IDF -> NB
evaluate(
    "Tokenizer->TF-IDF + MultinomialNB",
    MultinomialNBCalculator(),
    MultinomialNBApplier(),
    train_chain[chain_cols],
    test_chain[chain_cols],
    {"alpha": 0.1},
)

TF-IDF + MultinomialNB                   accuracy = 0.866
TF-IDF + LogisticRegression              accuracy = 0.847
Tokenizer->TF-IDF + MultinomialNB        accuracy = 0.892


0.8915502328675982

In [8]:
# Bernoulli NB wants "word present/absent" features -> feed it raw counts
# (it binarizes internally at the `binarize` threshold).
count_model_cfg = {"columns": ["text"], "max_features": 2000, "min_df": 2}
cvm_art = CountVectorizerCalculator().fit(train_df, count_model_cfg)
cvm_app = CountVectorizerApplier()
train_cvm = cvm_app.apply(train_df, cvm_art)
test_cvm = cvm_app.apply(test_df, cvm_art)
cvm_cols = cvm_art["output_columns"]

evaluate(
    "Count + BernoulliNB",
    BernoulliNBCalculator(),
    BernoulliNBApplier(),
    train_cvm[cvm_cols],
    test_cvm[cvm_cols],
    {"binarize": 0.0},
)

print("\nLeaderboard (higher is better):")
for name, acc in sorted(results.items(), key=lambda kv: kv[1], reverse=True):
    bar = "#" * int(acc * 30)
    print(f"  {acc:.3f}  {bar:<30}  {name}")

Count + BernoulliNB                      accuracy = 0.735

Leaderboard (higher is better):
  0.892  ##########################      Tokenizer->TF-IDF + MultinomialNB
  0.866  #########################       TF-IDF + MultinomialNB
  0.847  #########################       TF-IDF + LogisticRegression
  0.735  ######################          Count + BernoulliNB


## 7. Sentence Embedder — modern dense embeddings (optional)

Bag-of-words / TF-IDF treat "car" and "automobile" as completely unrelated. **Sentence embeddings** fix this: a pre-trained neural model maps each document to a dense vector where *similar meanings land near each other*. This often beats TF-IDF on short, nuanced text.

**Important details:**
- This needs the **optional** `sentence-transformers` package: `pip install skyulf[nlp]`. The cell skips gracefully if it's missing.
- The first run **downloads a model** (~90 MB for `all-MiniLM-L6-v2`) and caches it.
- Output is **dense and can be negative**, so pair it with **Logistic Regression**, *not* Naive Bayes.

**Use it when:** text is short (titles, reviews, tickets), meaning matters more than exact words, and you can afford the model download.

In [9]:
try:
    from skyulf.preprocessing.vectorization import (
        SentenceEmbedderApplier,
        SentenceEmbedderCalculator,
    )

    emb_cfg = {"columns": ["text"], "model_name": "all-MiniLM-L6-v2", "normalize": True}
    emb_art = SentenceEmbedderCalculator().fit(train_df, emb_cfg)
    emb_app = SentenceEmbedderApplier()
    train_emb = emb_app.apply(train_df, emb_art)
    test_emb = emb_app.apply(test_df, emb_art)
    emb_cols = emb_art["output_columns"]
    print(f"Each document is now a {emb_art['embedding_dim']}-dim vector.")

    # Dense + possibly negative -> Logistic Regression (NOT Naive Bayes)
    evaluate(
        "Embeddings + LogisticRegression",
        LogisticRegressionCalculator(),
        LogisticRegressionApplier(),
        train_emb[emb_cols],
        test_emb[emb_cols],
        {"max_iter": 1000},
    )
except ImportError as exc:
    print("SentenceEmbedder skipped - optional dependency not installed:", exc)
    print("Enable it with:  pip install skyulf[nlp]")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Each document is now a 384-dim vector.
Embeddings + LogisticRegression          accuracy = 0.940


## Cheat sheet — what to reach for

**Pick a vectorizer:**
- Starting out / want interpretability -> **TF-IDF Vectorizer** (best default).
- Need a dead-simple baseline -> **Count Vectorizer**.
- Dataset too big to store a vocabulary / streaming -> **Hashing Vectorizer**.
- Short text where *meaning* matters -> **Sentence Embedder** (`pip install skyulf[nlp]`).

**Pick a model:**
- TF-IDF or counts -> **Multinomial Naive Bayes** (fast, strong) or **Logistic Regression**.
- Binary word presence -> **Bernoulli Naive Bayes**.
- Dense embeddings -> **Logistic Regression** (never Naive Bayes — it needs non-negative inputs).

**The universal recipe:**
```python
art = SomeVectorizerCalculator().fit(train_df, config)   # learn on TRAIN only
train_X = SomeVectorizerApplier().apply(train_df, art)    # apply to train
test_X  = SomeVectorizerApplier().apply(test_df,  art)    # apply same transform to test
model = SomeModelCalculator().fit(train_X[art['output_columns']], train_df['label'])
preds = SomeModelApplier().predict(test_X[art['output_columns']], model)
```

**Chaining** (clean once, vectorize after) works because every node speaks the same DataFrame-in / DataFrame-out language:
```
text -> Tokenizer -> text__tokens -> TF-IDF -> features -> model
```

### In the visual canvas
These same nodes live in the **Preprocessing** section of the node sidebar. The **"Text Classification"** starter template wires `dataset -> Text Cleaning -> TF-IDF -> Train/Test Split -> Logistic Regression` for you — load it, point it at your text column, pick the label, and **Run All**.